In [ ]:
from snowflake.snowpark.functions import (col, lit, upper,
    call_udf, udtf, function, call_function, call_table_function)
from snowflake.snowpark.types import StructType, StructField, StringType
from snowflake.snowpark.context import get_active_session

In [ ]:
session = get_active_session()
df = session.create_dataframe(
    [[1, 'John Doe'], [2, 'Mary Poppins']],
    schema=["id", "name"])
df

In [ ]:
# call Snowpark wrapper to built-in SQL function
df.select("id", upper(col("name")).alias("upper_name"))

In [ ]:
# call explicit wrapper to built-in SQL function
df.select("id", function("upper")(col("name")).alias("upper_name"))

In [ ]:
# call directly built-in SQL function (~call_builtin)
df.select("id", call_function("upper", col("name")).alias("upper_name"))

In [ ]:
# call registered stored proc
session.call("procPython", 1)

In [ ]:
# call registered UDF
df.select(call_udf("udfPython", col("id")).alias("udf"), "name")

In [ ]:
# define new UDTF handler
class MyClass:
    def process(self, s):
        yield (s, )
        yield (s, )

# create new anonymous UDTF + call it
udtfPython = udtf(MyClass,
    output_schema=StructType([StructField("s", StringType())]),
    input_types=[StringType()])
session.table_function(udtfPython(lit("abc")))

In [ ]:
# call built-in SQL table function on constant value
session.table_function("split_to_table", lit("John Doe"), lit(" "))

In [ ]:
# call built-in SQL table function on constant value
session.table_function(call_table_function("split_to_table", lit("John Doe"), lit(" ")))

In [ ]:
# call built-in SQL table function on table values
df.join_table_function("split_to_table", col("name"), lit(" "))